In [4]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SyntheticDataGen") \
    .getOrCreate()

In [124]:
output_base = "/content/data/applications-gen/raw"

In [6]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
import datetime
import os

spark = SparkSession.builder \
    .appName("SyntheticDataGen") \
    .config("spark.sql.warehouse.dir", "file:/C:/temp") \
    .getOrCreate()

days = 7
sessions_per_day = 20000
users_count = 5000

real_apps = [
    "Chrome", "Edge", "Firefox",
    "Slack", "Discord", "Teams", "Outlook",
    "VSCode", "PyCharm", "IntelliJ",
    "Steam", "EpicGames", "Photoshop", "Illustrator"
]

power_modes = ["high", "balanced", "powersave"]
update_events = ["updated", "killed", "optimized"]

output_base = "../data/applications-gen/raw"

start_date = datetime.date.today() - datetime.timedelta(days=7)
dates = [start_date + datetime.timedelta(days=i) for i in range(days)]

def rand_ts(date):
    return datetime.datetime.combine(date, datetime.time()) + \
           datetime.timedelta(seconds=random.randint(0, 86400))

def random_ui_settings():
    return {
        "theme": random.choice(["light", "dark"]),
        "fontSize": str(random.choice([12, 14, 16])),
        "language": random.choice(["en", "de", "fr", "ru", "es"])
    }

all_users = [f"user_{i}" for i in range(users_count)]
all_devices = [f"device_{i}" for i in range(users_count)]


for date in dates:
    date_str = str(date)

    # -------------------------
    # 1) SESSIONS (uses real apps)
    # -------------------------
    sessions_rows = []
    for i in range(sessions_per_day):
        user = random.choice(all_users)
        device = random.choice(all_devices)

        ts_start = rand_ts(date)
        ts_end = ts_start + datetime.timedelta(seconds=random.randint(60, 3600))

        issue_type = random.choice([None, "very_long", "negative", "overlap"])
        if issue_type == "very_long":
            ts_end = ts_start + datetime.timedelta(hours=48)
        elif issue_type == "negative":
            ts_end = ts_start - datetime.timedelta(minutes=5)
        elif issue_type == "overlap":
            ts_start = ts_start - datetime.timedelta(minutes=10)

        duration = (ts_end - ts_start).total_seconds()

        sessions_rows.append((
            user,
            device,
            f"session_{date}_{i}",
            ts_start,
            ts_end,
            duration,
            [{"action": random.choice(["click","open","scroll"]),
              "ts": ts_start + datetime.timedelta(seconds=random.randint(0,300))}
             for _ in range(random.randint(1,5))],
            random.choice(real_apps),      # REAL APP NAME
            random.choice(power_modes),
            random.randint(5, 100),
            random_ui_settings()
        ))

    sessions_schema = StructType([
        StructField("userid", StringType()),
        StructField("deviceid", StringType()),
        StructField("sessionid", StringType()),
        StructField("clientTs", TimestampType()),
        StructField("serverTs", TimestampType()),
        StructField("sessionDuration", DoubleType()),
        StructField("actions", ArrayType(
            StructType([
                StructField("action", StringType()),
                StructField("ts", TimestampType())
            ])
        )),
        StructField("applicationName", StringType()),
        StructField("powerMode", StringType()),
        StructField("batteryPercent", IntegerType()),
        StructField("uiSettings", MapType(StringType(), StringType()))
    ])

    sessions_df = spark.createDataFrame(sessions_rows, sessions_schema)
    sessions_df.write.mode("overwrite").parquet(f"{output_base}/sessions/{date_str}")


    # detect devices today (70% of user base)
    detected_devices = random.sample(all_devices, int(users_count * 0.7))


    # -------------------------
    # 2) HARDWARE (key: deviceid)
    # -------------------------
    hardware_df = spark.createDataFrame(
        [(d,
          random.choice(["Intel i5", "Intel i7", "Ryzen 5", "Ryzen 7"]),
          random.choice([8, 16, 32]),
          random.choice(["Windows 10", "Windows 11", "Linux", "macOS"]),
          random.choice(["Laptop", "Desktop"])
         )
         for d in detected_devices],
        ["deviceid", "cpu", "ramGb", "os", "formFactor"]
    )
    hardware_df.write.mode("overwrite").parquet(f"{output_base}/hardware/{date_str}")


    # -------------------------
    # 3) APPLICATIONS (key: deviceid, MapType)
    # -------------------------
    apps_rows = []
    for d in detected_devices:
        app_map = {app: random.choice(["1.0", "2.0", "3.1", "4.0"])
                   for app in random.sample(real_apps, random.randint(3, 7))}
        apps_rows.append((d, app_map))

    applications_df = spark.createDataFrame(
        apps_rows,
        StructType([
            StructField("deviceid", StringType()),
            StructField("installedApps", MapType(StringType(), StringType()))
        ])
    )
    applications_df.write.mode("overwrite").parquet(f"{output_base}/applications/{date_str}")


    # -------------------------
    # 4) UPDATE EVENTS (key: deviceid)
    # -------------------------
    update_rows = []
    for d in detected_devices:
        for _ in range(random.randint(1,5)):
            update_rows.append((
                d,
                random.choice(real_apps),
                random.choice(update_events),
                rand_ts(date)
            ))

    update_df = spark.createDataFrame(
        update_rows,
        ["deviceid", "application", "eventType", "eventTs"]
    )
    update_df.write.mode("overwrite").parquet(f"{output_base}/updateEvents/{date_str}")

print("DONE.")

DONE.


In [7]:
for date in dates:
    date_str = str(date)

    detected_users = random.sample(all_users, int(users_count * 0.7))

    users_df = spark.createDataFrame(
        [(u, random.randint(18,70), random.choice(["US","DE","FR","AM","UK"]))
         for u in detected_users],
        ["userid", "age", "country"]
    )
    users_df.write.mode("overwrite").parquet(f"{output_base}/users/{date_str}")

**Installs (curated to events)**

Day 0

In [11]:
df = spark.read.parquet('../data/applications-gen/raw/sessions/2026-01-11')

In [13]:
df.show(5, truncate=False)

+-----------+----------------------------------------------------------------------------------------------------------+----------+
|deviceid   |installedApps                                                                                             |eventDate |
+-----------+----------------------------------------------------------------------------------------------------------+----------+
|device_3349|{PyCharm -> 2.0, Discord -> 4.0, Slack -> 3.1, Photoshop -> 1.0, Firefox -> 2.0}                          |2026-01-11|
|device_2974|{IntelliJ -> 3.1, PyCharm -> 1.0, Outlook -> 1.0, Steam -> 3.1, Teams -> 2.0, Discord -> 4.0, Edge -> 4.0}|2026-01-11|
|device_3574|{Slack -> 2.0, Illustrator -> 3.1, Outlook -> 1.0}                                                        |2026-01-11|
|device_1350|{Slack -> 3.1, Steam -> 4.0, Teams -> 1.0, Chrome -> 4.0, Discord -> 2.0, Edge -> 3.1}                    |2026-01-11|
|device_1778|{PyCharm -> 4.0, Slack -> 3.1, Illustrator -> 3.1}             

In [14]:
from pyspark.sql import functions as F
date = '2026-01-11'
df = spark.read.parquet(f'../data/applications-gen/raw/applications/{date}')\
    .withColumn('eventDate', F.split(F.input_file_name(), '/')[7])

In [15]:
df.printSchema()

root
 |-- deviceid: string (nullable = true)
 |-- installedApps: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- eventDate: string (nullable = true)



In [16]:
df.show(5, truncate=False)

+-----------+----------------------------------------------------------------------------------------------------------+----------+
|deviceid   |installedApps                                                                                             |eventDate |
+-----------+----------------------------------------------------------------------------------------------------------+----------+
|device_3349|{PyCharm -> 2.0, Discord -> 4.0, Slack -> 3.1, Photoshop -> 1.0, Firefox -> 2.0}                          |2026-01-11|
|device_2974|{IntelliJ -> 3.1, PyCharm -> 1.0, Outlook -> 1.0, Steam -> 3.1, Teams -> 2.0, Discord -> 4.0, Edge -> 4.0}|2026-01-11|
|device_3574|{Slack -> 2.0, Illustrator -> 3.1, Outlook -> 1.0}                                                        |2026-01-11|
|device_1350|{Slack -> 3.1, Steam -> 4.0, Teams -> 1.0, Chrome -> 4.0, Discord -> 2.0, Edge -> 3.1}                    |2026-01-11|
|device_1778|{PyCharm -> 4.0, Slack -> 3.1, Illustrator -> 3.1}             

In [17]:
events = df.select('*', F.explode('installedApps'))\
    .withColumnRenamed('key', 'application')\
    .withColumnRenamed('value', 'version')\
    .drop('installedApps')\
    .withColumn('eventType', F.lit('install'))

In [18]:
events.write.mode('overwrite').parquet(f'../data/applications-gen/curated/install-events/{date}')

In [19]:
events.count()

17496

In [20]:
df = spark.read.parquet(f'../data/applications-gen/curated/install-events/{date}')

In [21]:
df.count()

17496

In [22]:
df.drop('eventType').write.mode('overwrite').parquet(f'../data/applications-gen/curated/installed-states/{date}')

Day 1+

In [23]:
date = '2026-01-12'
prev_date = '2026-01-11'

In [24]:
d0_states = spark.read.parquet(f'../data/applications-gen/curated/installed-states/{prev_date}')
d1_raw = spark.read.parquet(f'../data/applications-gen/raw/applications/{date}')

In [25]:
d1_raw = d1_raw.select('*', F.explode('installedApps'))\
    .withColumnRenamed('key', 'application')\
    .withColumnRenamed('value', 'version')\
    .withColumn('eventDate', F.split(F.input_file_name(), '/')[5])\
    .drop('installedApps')

In [26]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


print(f"Processing data for date: {date}, using previous state from: {prev_date}")

d0_states = spark.read.parquet(f'../data/applications-gen/curated/installed-states/{prev_date}')
print(f"Loaded d0_states for {prev_date}. Count: {d0_states.count()}")

d1_raw = spark.read.parquet(f'../data/applications-gen/raw/applications/{date}')
print(f"Loaded raw applications for {date}. Count: {d1_raw.count()}")

d1_raw_transformed = d1_raw.select('*', F.explode('installedApps'))\
    .withColumnRenamed('key', 'application')\
    .withColumnRenamed('value', 'version')\
    .withColumn('eventDate', F.split(F.input_file_name(), '/')[5])\
    .drop('installedApps')
print(f"Transformed d1_raw for {date}. Count: {d1_raw_transformed.count()}")

d1_full = d0_states.unionByName(d1_raw_transformed)
print(f"Union of d0_states and d1_raw_transformed. Total count: {d1_full.count()}")

w = Window().partitionBy('deviceid', 'application').orderBy(F.col('eventDate').desc())
d1_full = d1_full.withColumn('occOrder', F.row_number().over(w))
print(f"Applied window function. d1_full count: {d1_full.count()}")

d1_states_curated = d1_full.filter(F.col('occOrder') == 1).drop('occOrder')
print(f"Filtered for latest states. Final curated states count for {date}: {d1_states_curated.count()}")

d1_states_curated.write.mode('overwrite').parquet(f'../data/applications-gen/curated/installed-states/{date}')
print(f"Curated installed states for {date} saved.")

Processing data for date: 2026-01-12, using previous state from: 2026-01-11
Loaded d0_states for 2026-01-11. Count: 17496
Loaded raw applications for 2026-01-12. Count: 3500
Transformed d1_raw for 2026-01-12. Count: 17506
Union of d0_states and d1_raw_transformed. Total count: 35002
Applied window function. d1_full count: 35002
Filtered for latest states. Final curated states count for 2026-01-12: 30671
Curated installed states for 2026-01-12 saved.


In [27]:
d0_states.show(5)

+-----------+----------+-----------+-------+
|   deviceid| eventDate|application|version|
+-----------+----------+-----------+-------+
|device_3349|2026-01-11|    PyCharm|    2.0|
|device_3349|2026-01-11|    Discord|    4.0|
|device_3349|2026-01-11|      Slack|    3.1|
|device_3349|2026-01-11|  Photoshop|    1.0|
|device_3349|2026-01-11|    Firefox|    2.0|
+-----------+----------+-----------+-------+
only showing top 5 rows


In [28]:
d1_full.show(5)

+--------+----------+-----------+-------+--------+
|deviceid| eventDate|application|version|occOrder|
+--------+----------+-----------+-------+--------+
|device_0|2026-01-11|       Edge|    3.1|       1|
|device_0|2026-01-11|    Firefox|    4.0|       1|
|device_0|2026-01-11|Illustrator|    4.0|       1|
|device_0|2026-01-11|   IntelliJ|    3.1|       1|
|device_0|2026-01-11|  Photoshop|    1.0|       1|
+--------+----------+-----------+-------+--------+
only showing top 5 rows


In [29]:
w = Window().partitionBy('deviceid', 'application').orderBy(F.col('eventDate').desc())
d1_full = d1_full.withColumn('occOrder', F.row_number().over(w))

In [30]:
# 1. d0+ d1+ - 1,2              - YES       - no-event
# 1. d0+ d1- - 1 d0 date        - NO        - 1. uninstall 2. no-event
# 1. d0- d1+ - 1 d1 date        - YES       - install

In [31]:
d1_full = d1_full.filter((F.col('occOrder')==2) | ((F.col('occOrder')==1) & (F.col('eventDate')==date)))

In [32]:
d1_full.write.mode('overwrite').parquet(f'../data/applications-gen/curated/installed-states/{date}')

In [33]:
# 1. map and array operations
# 2. rowNumber()

In [34]:
d0_raw = spark.read.parquet(f'../data/applications-gen/raw/applications/{prev_date}')
d1_raw = spark.read.parquet(f'../data/applications-gen/raw/applications/{date}')

In [35]:
d0_raw = d0_raw.withColumn('apps_prev', F.map_keys('installedApps'))
d1_raw = d1_raw.withColumn('apps', F.map_keys('installedApps'))

df = d0_raw.join(d1_raw, how='outer', on='deviceId')

In [36]:
df\
    .withColumn('intersect', F.array_intersect('apps', 'apps_prev'))\
    .withColumn('install', F.array_except('apps', 'intersect'))\
    .withColumn('uninstall', F.array_except('apps_prev', 'intersect'))\
    .show(10, False, True)

-RECORD 0-----------------------------------------------------------------------------------------------------------------------------
 deviceid      | device_0                                                                                                             
 installedApps | {IntelliJ -> 3.1, PyCharm -> 1.0, Illustrator -> 4.0, Teams -> 2.0, Photoshop -> 1.0, Edge -> 3.1, Firefox -> 4.0}   
 apps_prev     | [IntelliJ, PyCharm, Illustrator, Teams, Photoshop, Edge, Firefox]                                                    
 installedApps | NULL                                                                                                                 
 apps          | NULL                                                                                                                 
 intersect     | NULL                                                                                                                 
 install       | NULL                                  

In [37]:
# OVERALL BEST SOLUTION - Window functions (lag/lead) to be used with complex data types
# 1. one row per device  - deviceid | timestamp | applications (map | array)
# 2. one row per device  - deviceid | timestamp | apps | prev_timestamp | prev_apps
# 3. store states as arrays | maps
# 4. a. timestamp is not null and prev_timestamp is not null - CLEAR
# 4. b. timestamp is not null and next_timestamp is not null - no install events and no uninstall events
# 4. c. timestamp is not null and prev_timestamp is not null - all install events and no uninstall events

# Task
Implement a daily data processing pipeline for "applications-gen" data, including functions for enriching `userDevice` state, aggregating total users and application installs/uninstalls, and running data quality checks, then orchestrate these functions to process data iteratively for each day.

In [46]:
import os

enriched_base = "../data/applications-gen/enriched"
aggregated_base = "../data/applications-gen/aggregated"

os.makedirs(enriched_base, exist_ok=True)
os.makedirs(aggregated_base, exist_ok=True)

print(f"Enriched data directory created: {enriched_base}")
print(f"Aggregated data directory created: {aggregated_base}")

Enriched data directory created: ../data/applications-gen/enriched
Aggregated data directory created: ../data/applications-gen/aggregated


## Create Day-0 userDevice State

### Subtask:
Calculate the initial state of the `userDevice` table for the first day of data (`dates[0]`). This involves reading raw `sessions`, `hardware`, `applications`, and `updateEvents` data for Day 0, joining them to form a base DataFrame, and aggregating the information to create a single row per unique `(userId, deviceId)` pair, including `registerTs`, `lastActivityTs`, `lastSessionTs`, `applications` (map), and `hardware` (array). The result will be saved in the enriched data path.


In [47]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# 1. Extract the string representation of the first date from the dates list
current_date_str = dates[0].strftime('%Y-%m-%d')
print(f"Processing Day-0 for date: {current_date_str}")

# 2. Load raw data for current_date_str
raw_sessions_df = spark.read.parquet(f"{output_base}/sessions/{current_date_str}")
raw_hardware_df = spark.read.parquet(f"{output_base}/hardware/{current_date_str}")
raw_applications_df = spark.read.parquet(f"{output_base}/applications/{current_date_str}")
raw_update_events_df = spark.read.parquet(f"{output_base}/updateEvents/{current_date_str}")
raw_users_df = spark.read.parquet(f"{output_base}/users/{current_date_str}") # Loaded but not used in final join as per instructions

print("Raw data loaded.")

# 3. Create an initial userDevice base DataFrame
user_device_base_df = raw_sessions_df.groupBy('userid', 'deviceid').agg(
    F.min('clientTs').alias('registerTs'),
    F.max('serverTs').alias('lastActivityTs'),
    F.max('serverTs').alias('lastSessionTs')
)
print("Initial userDevice base DataFrame created.")

# 4. Prepare hardware information
hardware_agg_df = raw_hardware_df.withColumn(
    'hardware_array',
    F.array(
        F.col('cpu'),
        F.col('ramGb').cast(StringType()),
        F.col('os'),
        F.col('formFactor')
    )
).select('deviceid', 'hardware_array')
print("Hardware aggregated.")

# 5. Join user_device_base_df with hardware_agg_df
user_device_base_df = user_device_base_df.join(hardware_agg_df, on='deviceid', how='left_outer')
print("Joined with hardware.")

# 6. Prepare applications information
applications_map_df = raw_applications_df.withColumnRenamed('installedApps', 'applications_map')\
    .select('deviceid', 'applications_map')
print("Applications map prepared.")

# 7. Join the current user_device_base_df with applications_map_df
user_device_base_df = user_device_base_df.join(applications_map_df, on='deviceid', how='left_outer')
print("Joined with applications.")

# 8. Prepare update events information
update_agg_df = raw_update_events_df.groupBy('deviceid').agg(
    F.max('eventTs').alias('maxUpdateTs')
)
print("Update events aggregated.")

# 9. Join and update lastActivityTs
user_device_base_df = user_device_base_df.join(update_agg_df, on='deviceid', how='left_outer')\
    .withColumn('lastActivityTs', F.greatest(F.col('lastActivityTs'), F.col('maxUpdateTs')))\
    .drop('maxUpdateTs')
print("Joined with update events and updated lastActivityTs.")

# 10. Select final columns for the Day-0 userDevice state DataFrame
day0_user_device_df = user_device_base_df.select(
    F.col('userid').alias('userId'),
    F.col('deviceid').alias('deviceId'),
    'registerTs',
    'lastActivityTs',
    'lastSessionTs',
    F.col('applications_map').alias('applications'),
    F.col('hardware_array').alias('hardware')
)

print("Final Day-0 userDevice DataFrame prepared.")

# 11. Save day0_user_device_df
output_path = f"{enriched_base}/userDevice/{current_date_str}"
day0_user_device_df.write.mode('overwrite').parquet(output_path)

print(f"Day-0 userDevice state saved to: {output_path}")
day0_user_device_df.show(5, truncate=False)

Processing Day-0 for date: 2026-01-11
Raw data loaded.
Initial userDevice base DataFrame created.
Hardware aggregated.
Joined with hardware.
Applications map prepared.
Joined with applications.
Update events aggregated.
Joined with update events and updated lastActivityTs.
Final Day-0 userDevice DataFrame prepared.
Day-0 userDevice state saved to: ../data/applications-gen/enriched/userDevice/2026-01-11
+---------+-----------+-------------------+-------------------+-------------------+---------------------------------------------------------------------------------------------------+----------------------------------+
|userId   |deviceId   |registerTs         |lastActivityTs     |lastSessionTs      |applications                                                                                       |hardware                          |
+---------+-----------+-------------------+-------------------+-------------------+-------------------------------------------------------------------------

## Define enrich_user_device_daily Function

Implement a Python function `enrich_user_device_daily` that takes `current_date_str` and `prev_date_str` as arguments. This function will load the `userDevice` state from the previous day, along with the current day's raw `sessions`, `hardware`, `applications`, and `updateEvents` data. It will then merge this information, updating existing `userDevice` entries and creating new ones as necessary, to produce the `userDevice` state for the `current_date_str`. The function will save the updated `userDevice` DataFrame to the enriched data path.


In [48]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

def enrich_user_device_daily(spark, current_date_str, prev_date_str, output_base, enriched_base):
    print(f"Enriching userDevice for {current_date_str} based on {prev_date_str}")

    # 1. Load previous day's userDevice state
    prev_day_user_device_df = spark.read.parquet(f"{enriched_base}/userDevice/{prev_date_str}")
    print(f"Loaded previous day's userDevice state for {prev_date_str}. Count: {prev_day_user_device_df.count()}")

    # 2. Load current day's raw data
    raw_sessions_df = spark.read.parquet(f"{output_base}/sessions/{current_date_str}")
    raw_hardware_df = spark.read.parquet(f"{output_base}/hardware/{current_date_str}")
    raw_applications_df = spark.read.parquet(f"{output_base}/applications/{current_date_str}")
    raw_update_events_df = spark.read.parquet(f"{output_base}/updateEvents/{current_date_str}")
    raw_users_df = spark.read.parquet(f"{output_base}/users/{current_date_str}") # Loaded for completeness, though not directly used in the join for current_day_enriched_df
    print(f"Loaded current day's raw data for {current_date_str}.")

    # 3. Create a base DataFrame for the current day's sessions
    current_day_session_summary_df = raw_sessions_df.groupBy('userid', 'deviceid').agg(
        F.min('clientTs').alias('registerTs'),
        F.max('serverTs').alias('lastActivityTs'),
        F.max('serverTs').alias('lastSessionTs')
    )
    print(f"Created current day session summary. Count: {current_day_session_summary_df.count()}")

    # 4. Prepare current day's hardware information
    hardware_agg_df = raw_hardware_df.withColumn(
        'hardware_array',
        F.array(
            F.col('cpu'),
            F.col('ramGb').cast(StringType()),
            F.col('os'),
            F.col('formFactor')
        )
    ).select('deviceid', 'hardware_array')
    print(f"Prepared hardware information. Count: {hardware_agg_df.count()}")

    # 5. Prepare current day's applications information
    applications_map_df = raw_applications_df.withColumnRenamed('installedApps', 'applications_map')\
        .select('deviceid', 'applications_map')
    print(f"Prepared applications information. Count: {applications_map_df.count()}")

    # 6. Prepare current day's update events information
    update_agg_df = raw_update_events_df.groupBy('deviceid').agg(
        F.max('eventTs').alias('maxUpdateTs')
    )
    print(f"Prepared update events information. Count: {update_agg_df.count()}")

    # 7. Join current_day_session_summary_df with hardware_agg_df, applications_map_df, and update_agg_df
    current_day_enriched_df = current_day_session_summary_df \
        .join(hardware_agg_df, on='deviceid', how='left_outer') \
        .join(applications_map_df, on='deviceid', how='left_outer') \
        .join(update_agg_df, on='deviceid', how='left_outer') \
        .withColumn('lastActivityTs', F.greatest(F.col('lastActivityTs'), F.col('maxUpdateTs'))) \
        .drop('maxUpdateTs')
    print(f"Joined current day's data. Count: {current_day_enriched_df.count()}")

    # 8. Align the schema of current_day_enriched_df with prev_day_user_device_df
    current_day_enriched_df = current_day_enriched_df.select(
        F.col('userid').alias('userId'),
        F.col('deviceid').alias('deviceId'),
        'registerTs',
        'lastActivityTs',
        'lastSessionTs',
        F.col('applications_map').alias('applications'),
        F.col('hardware_array').alias('hardware')
    )
    print("Aligned current day enriched DataFrame schema.")

    # 9. Union prev_day_user_device_df with the prepared current_day_enriched_df
    union_df = prev_day_user_device_df.unionByName(current_day_enriched_df)
    print(f"Union of previous and current day data. Total count: {union_df.count()}")

    # 10. Use a window function to get the latest state for each (userId, deviceId) pair
    window_spec = Window().partitionBy('userId', 'deviceId').orderBy(F.col('lastActivityTs').desc())
    current_day_user_device_df = union_df.withColumn('occOrder', F.row_number().over(window_spec))
    current_day_user_device_df = current_day_user_device_df.filter(F.col('occOrder') == 1).drop('occOrder')
    print(f"Filtered for latest states. Final current day userDevice count: {current_day_user_device_df.count()}")

    # 11. Save the current_day_user_device_df
    output_path = f"{enriched_base}/userDevice/{current_date_str}"
    current_day_user_device_df.write.mode('overwrite').parquet(output_path)

    print(f"Successfully enriched and saved userDevice state for {current_date_str} to {output_path}")
    # Return the DataFrame for potential further use in the notebook
    return current_day_user_device_df

print("Defined enrich_user_device_daily function.")

Defined enrich_user_device_daily function.


**Reasoning**:
The next logical step is to call the `enrich_user_device_daily` function for each subsequent day (Day 1 onwards) to process the data iteratively and build up the enriched `userDevice` state over time.



In [50]:
print("Starting daily enrichment process...")

# Loop through dates starting from the second day (dates[1])
for i in range(1, len(dates)):
    current_date_str = dates[i].strftime('%Y-%m-%d')
    prev_date_str = dates[i-1].strftime('%Y-%m-%d')

    # Call the enrichment function for the current day
    enriched_df_for_day = enrich_user_device_daily(spark, current_date_str, prev_date_str, output_base, enriched_base)
    print(f"Enrichment for {current_date_str} completed.")

print("Daily enrichment process finished.")

Starting daily enrichment process...
Enriching userDevice for 2026-01-12 based on 2026-01-11
Loaded previous day's userDevice state for 2026-01-11. Count: 19997
Loaded current day's raw data for 2026-01-12.
Created current day session summary. Count: 19989
Prepared hardware information. Count: 3500
Prepared applications information. Count: 3500
Prepared update events information. Count: 3500
Joined current day's data. Count: 19989
Aligned current day enriched DataFrame schema.
Union of previous and current day data. Total count: 39986
Filtered for latest states. Final current day userDevice count: 39963
Successfully enriched and saved userDevice state for 2026-01-12 to ../data/applications-gen/enriched/userDevice/2026-01-12
Enrichment for 2026-01-12 completed.
Enriching userDevice for 2026-01-13 based on 2026-01-12
Loaded previous day's userDevice state for 2026-01-12. Count: 39963
Loaded current day's raw data for 2026-01-13.
Created current day session summary. Count: 19994
Prepared 

## Define aggregate_total_n_of_users_daily Function

Implement a Python function `aggregate_total_n_of_users_daily` that takes `current_date_str` as an argument. This function will read all `users` data from the very first date (`dates[0]`) up to and including the `current_date_str`, calculate the cumulative number of distinct users observed so far, and save this aggregate for the `current_date_str` in the aggregated data path.


In [51]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType

def aggregate_total_n_of_users_daily(spark, current_date_str, output_base, aggregated_base, dates):
    print(f"Aggregating total distinct users for date: {current_date_str}")

    # 1. Initialize an empty list to store user DataFrames
    user_dfs = []

    # 2. Iterate through the dates list and collect user DataFrames up to current_date_str
    for dt in dates:
        dt_str = dt.strftime('%Y-%m-%d')
        if dt_str <= current_date_str:
            try:
                user_df_path = f"{output_base}/users/{dt_str}"
                user_df = spark.read.parquet(user_df_path).select('userid')
                user_dfs.append(user_df)
                print(f"  Loaded users for {dt_str}")
            except Exception as e:
                print(f"  Warning: Could not load users for {dt_str}. Error: {e}")

    # 3. Union all collected user DataFrames into a single DataFrame
    if not user_dfs:
        print(f"No user data found up to {current_date_str}. Creating empty DataFrame.")
        total_distinct_users = 0
    else:
        # Union all user DataFrames
        unioned_users_df = user_dfs[0]
        for i in range(1, len(user_dfs)):
            unioned_users_df = unioned_users_df.union(user_dfs[i])

        # 4. Count distinct userids
        total_distinct_users = unioned_users_df.select('userid').distinct().count()
        print(f"  Total distinct users up to {current_date_str}: {total_distinct_users}")

    # 5. Create a new Spark DataFrame for the aggregated result
    schema = StructType([
        StructField("eventDate", StringType(), True),
        StructField("totalDistinctUsers", LongType(), True)
    ])

    aggregated_result_df = spark.createDataFrame(
        [(current_date_str, total_distinct_users)],
        schema=schema
    )

    # 6. Define the output path for the aggregated data
    output_path = f"{aggregated_base}/total_users/{current_date_str}"

    # 7. Save the resulting DataFrame
    aggregated_result_df.write.mode('overwrite').parquet(output_path)

    print(f"Aggregated total distinct users for {current_date_str} saved to: {output_path}")
    return aggregated_result_df

print("Defined aggregate_total_n_of_users_daily function.")

Defined aggregate_total_n_of_users_daily function.


In [52]:
print("Starting daily aggregation of total distinct users...")

# Loop through each date to perform daily aggregation
for current_date in dates:
    current_date_str = current_date.strftime('%Y-%m-%d')

    # Call the aggregation function for the current day
    aggregated_users_df = aggregate_total_n_of_users_daily(spark, current_date_str, output_base, aggregated_base, dates)
    print(f"Aggregation for {current_date_str} completed.")

print("Daily aggregation of total distinct users finished.")

Starting daily aggregation of total distinct users...
Aggregating total distinct users for date: 2026-01-11
  Loaded users for 2026-01-11
  Total distinct users up to 2026-01-11: 3500
Aggregated total distinct users for 2026-01-11 saved to: ../data/applications-gen/aggregated/total_users/2026-01-11
Aggregation for 2026-01-11 completed.
Aggregating total distinct users for date: 2026-01-12
  Loaded users for 2026-01-11
  Loaded users for 2026-01-12
  Total distinct users up to 2026-01-12: 4563
Aggregated total distinct users for 2026-01-12 saved to: ../data/applications-gen/aggregated/total_users/2026-01-12
Aggregation for 2026-01-12 completed.
Aggregating total distinct users for date: 2026-01-13
  Loaded users for 2026-01-11
  Loaded users for 2026-01-12
  Loaded users for 2026-01-13
  Total distinct users up to 2026-01-13: 4869
Aggregated total distinct users for 2026-01-13 saved to: ../data/applications-gen/aggregated/total_users/2026-01-13
Aggregation for 2026-01-13 completed.
Aggr

## Define aggregate_n_of_installs_uninstalls_daily Function

Implement a Python function `aggregate_n_of_installs_uninstalls_daily` that takes `current_date_str` and `prev_date_str` as arguments. This function will load the raw `applications` data for both the `prev_date_str` and `current_date_str`, compare the installed applications for each device, and calculate the daily counts of application installs and uninstalls. The results will be saved in the aggregated data path.


In [55]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

def aggregate_n_of_installs_uninstalls_daily(spark, current_date_str, prev_date_str, output_base, aggregated_base):
    print(f"Aggregating installs/uninstalls for {current_date_str} compared to {prev_date_str}")

    # 1. Load raw applications data for previous and current dates
    # Handle cases where the raw applications file for prev_date_str might not exist
    try:
        prev_apps_df = spark.read.parquet(f"{output_base}/applications/{prev_date_str}")\
            .withColumnRenamed('installedApps', 'installedApps_prev')
    except Exception as e:
        print(f"Warning: No raw applications data for {prev_date_str}. Assuming no previous installations. Error: {e}")
        # Create an empty DataFrame with the expected schema if file not found
        prev_apps_schema = StructType([
            StructField("deviceid", StringType(), True),
            StructField("installedApps_prev", F.MapType(StringType(), StringType()), True)
        ])
        prev_apps_df = spark.createDataFrame([], schema=prev_apps_schema)

    try:
        curr_apps_df = spark.read.parquet(f"{output_base}/applications/{current_date_str}")\
            .withColumnRenamed('installedApps', 'installedApps_curr')
    except Exception as e:
        print(f"Error: No raw applications data for {current_date_str}. Cannot proceed with aggregation. Error: {e}")
        return

    # 2. Extract application names into array columns
    # Handle case where installedApps_prev might be null due to empty prev_apps_df
    prev_apps_df_processed = prev_apps_df.withColumn('apps_prev', F.when(F.col('installedApps_prev').isNull(), F.array()).otherwise(F.map_keys('installedApps_prev')))
    curr_apps_df_processed = curr_apps_df.withColumn('apps_curr', F.map_keys('installedApps_curr'))

    # 3. Perform a full outer join on deviceid
    joined_df = prev_apps_df_processed.alias('prev') \
        .join(curr_apps_df_processed.alias('curr'), on='deviceid', how='fullouter')

    # Fill nulls for apps_prev and apps_curr after join for correct array_except behavior
    joined_df = joined_df.withColumn('apps_prev_filled', F.coalesce(F.col('prev.apps_prev'), F.array())) \
                         .withColumn('apps_curr_filled', F.coalesce(F.col('curr.apps_curr'), F.array()))

    # 4. Calculate installed applications
    install_events = joined_df.withColumn('install_list', F.array_except(F.col('apps_curr_filled'), F.col('apps_prev_filled'))) \
        .filter(F.size('install_list') > 0) \
        .select(F.explode('install_list').alias('application'), F.lit('install').alias('eventType'))

    # 5. Calculate uninstalled applications
    uninstall_events = joined_df.withColumn('uninstall_list', F.array_except(F.col('apps_prev_filled'), F.col('apps_curr_filled'))) \
        .filter(F.size('uninstall_list') > 0) \
        .select(F.explode('uninstall_list').alias('application'), F.lit('uninstall').alias('eventType'))

    # 6. Union the install events and uninstall events DataFrames
    # 7. Add an eventDate column
    if install_events.count() == 0 and uninstall_events.count() == 0:
        print(f"No install or uninstall events for {current_date_str}.")
        # Create an empty DataFrame with the expected schema
        events_schema = StructType([
            StructField("application", StringType(), True),
            StructField("eventType", StringType(), True),
            StructField("eventDate", StringType(), True),
            StructField("eventCount", IntegerType(), True)
        ])
        daily_summary_df = spark.createDataFrame([], schema=events_schema)
    else:
        all_events = install_events.unionByName(uninstall_events)
        all_events = all_events.withColumn('eventDate', F.lit(current_date_str))

        # 8. Group by application, eventType, and eventDate, and count the occurrences
        daily_summary_df = all_events.groupBy('application', 'eventType', 'eventDate') \
            .agg(F.count(F.lit(1)).alias('eventCount'))

    # 9. Save the resulting aggregated DataFrame
    output_path = f"{aggregated_base}/installs_uninstalls/{current_date_str}"
    daily_summary_df.write.mode('overwrite').parquet(output_path)

    print(f"Aggregated install/uninstall events for {current_date_str} saved to: {output_path}")
    return daily_summary_df

print("Defined aggregate_n_of_installs_uninstalls_daily function.")

Defined aggregate_n_of_installs_uninstalls_daily function.


In [56]:
print("Starting daily aggregation of installs/uninstalls...")

# Loop through dates starting from the second day (dates[1])
for i in range(1, len(dates)):
    current_date_str = dates[i].strftime('%Y-%m-%d')
    prev_date_str = dates[i-1].strftime('%Y-%m-%d')

    # Call the aggregation function for the current day
    aggregated_installs_uninstalls_df = aggregate_n_of_installs_uninstalls_daily(spark, current_date_str, prev_date_str, output_base, aggregated_base)
    print(f"Aggregation for {current_date_str} completed.")

print("Daily aggregation of installs/uninstalls finished.")

Starting daily aggregation of installs/uninstalls...
Aggregating installs/uninstalls for 2026-01-12 compared to 2026-01-11
Aggregated install/uninstall events for 2026-01-12 saved to: ../data/applications-gen/aggregated/installs_uninstalls/2026-01-12
Aggregation for 2026-01-12 completed.
Aggregating installs/uninstalls for 2026-01-13 compared to 2026-01-12
Aggregated install/uninstall events for 2026-01-13 saved to: ../data/applications-gen/aggregated/installs_uninstalls/2026-01-13
Aggregation for 2026-01-13 completed.
Aggregating installs/uninstalls for 2026-01-14 compared to 2026-01-13
Aggregated install/uninstall events for 2026-01-14 saved to: ../data/applications-gen/aggregated/installs_uninstalls/2026-01-14
Aggregation for 2026-01-14 completed.
Aggregating installs/uninstalls for 2026-01-15 compared to 2026-01-14
Aggregated install/uninstall events for 2026-01-15 saved to: ../data/applications-gen/aggregated/installs_uninstalls/2026-01-15
Aggregation for 2026-01-15 completed.
Agg

## Define run_data_quality_checks Function

Implement a Python function `run_data_quality_checks` that performs data quality checks on raw datasets for a given date and reports the findings.


In [57]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, MapType, ArrayType

def run_data_quality_checks(spark, date_str, output_base):
    print(f"--- Running data quality checks for {date_str} ---")

    # Check 1: Sessions with negative duration
    try:
        sessions_df = spark.read.parquet(f"{output_base}/sessions/{date_str}")
        negative_duration_sessions = sessions_df.filter(F.col('sessionDuration') < 0).count()
        print(f"  Sessions with negative duration: {negative_duration_sessions}")
    except Exception as e:
        print(f"  Error loading sessions data for {date_str} for quality check: {e}")

    # Check 2: Devices with missing CPU or OS in hardware data
    try:
        hardware_df = spark.read.parquet(f"{output_base}/hardware/{date_str}")
        missing_hardware_info = hardware_df.filter(
            (F.col('cpu').isNull()) | (F.length(F.trim(F.col('cpu'))) == 0) |
            (F.col('os').isNull()) | (F.length(F.trim(F.col('os'))) == 0)
        ).count()
        print(f"  Devices with missing CPU or OS info: {missing_hardware_info}")
    except Exception as e:
        print(f"  Error loading hardware data for {date_str} for quality check: {e}")

    # Check 3: Devices with empty installed applications map
    try:
        applications_df = spark.read.parquet(f"{output_base}/applications/{date_str}")
        empty_applications_map = applications_df.filter(F.size(F.col('installedApps')) == 0).count()
        print(f"  Devices with empty installed applications: {empty_applications_map}")
    except Exception as e:
        print(f"  Error loading applications data for {date_str} for quality check: {e}")

    print(f"--- Finished data quality checks for {date_str} ---")

print("Defined run_data_quality_checks function.")

Defined run_data_quality_checks function.


In [58]:
print("Starting daily data quality checks...")

# Loop through each date to perform daily data quality checks
for current_date in dates:
    current_date_str = current_date.strftime('%Y-%m-%d')

    # Call the data quality check function for the current day
    run_data_quality_checks(spark, current_date_str, output_base)

print("Daily data quality checks finished.")

Starting daily data quality checks...
--- Running data quality checks for 2026-01-11 ---
  Sessions with negative duration: 5095
  Devices with missing CPU or OS info: 0
  Devices with empty installed applications: 0
--- Finished data quality checks for 2026-01-11 ---
--- Running data quality checks for 2026-01-12 ---
  Sessions with negative duration: 4962
  Devices with missing CPU or OS info: 0
  Devices with empty installed applications: 0
--- Finished data quality checks for 2026-01-12 ---
--- Running data quality checks for 2026-01-13 ---
  Sessions with negative duration: 4961
  Devices with missing CPU or OS info: 0
  Devices with empty installed applications: 0
--- Finished data quality checks for 2026-01-13 ---
--- Running data quality checks for 2026-01-14 ---
  Sessions with negative duration: 5048
  Devices with missing CPU or OS info: 0
  Devices with empty installed applications: 0
--- Finished data quality checks for 2026-01-14 ---
--- Running data quality checks for 20

## Orchestrate Daily Data Processing

Create a loop that iterates through the `dates` list, starting from the second day (since Day 0 is already handled). In each iteration, it will call the `enrich_user_device_daily` function to update the `userDevice` state, then the aggregation functions (`aggregate_total_n_of_users_daily` and `aggregate_n_of_installs_uninstalls_daily`), and finally the `run_data_quality_checks` function for the current day's data.


In [59]:
print("Starting daily data processing orchestration...")

# Loop through dates starting from the second day (index 1) as Day 0 is already handled
for i in range(1, len(dates)):
    current_date_str = dates[i].strftime('%Y-%m-%d')
    prev_date_str = dates[i-1].strftime('%Y-%m-%d')

    print(f"\n--- Processing data for {current_date_str} (previous day: {prev_date_str}) ---")

    # 1. Enrich userDevice state
    print(f"  Calling enrich_user_device_daily for {current_date_str}...")
    enrich_user_device_daily(spark, current_date_str, prev_date_str, output_base, enriched_base)
    print(f"  Finished enrich_user_device_daily for {current_date_str}.")

    # 2. Aggregate total number of users
    print(f"  Calling aggregate_total_n_of_users_daily for {current_date_str}...")
    aggregate_total_n_of_users_daily(spark, current_date_str, output_base, aggregated_base, dates)
    print(f"  Finished aggregate_total_n_of_users_daily for {current_date_str}.")

    # 3. Aggregate installs and uninstalls
    print(f"  Calling aggregate_n_of_installs_uninstalls_daily for {current_date_str}...")
    aggregate_n_of_installs_uninstalls_daily(spark, current_date_str, prev_date_str, output_base, aggregated_base)
    print(f"  Finished aggregate_n_of_installs_uninstalls_daily for {current_date_str}.")

    # 4. Run data quality checks
    print(f"  Calling run_data_quality_checks for {current_date_str}...")
    run_data_quality_checks(spark, current_date_str, output_base)
    print(f"  Finished run_data_quality_checks for {current_date_str}.")

print("\nDaily data processing orchestration finished.")

Starting daily data processing orchestration...

--- Processing data for 2026-01-12 (previous day: 2026-01-11) ---
  Calling enrich_user_device_daily for 2026-01-12...
Enriching userDevice for 2026-01-12 based on 2026-01-11
Loaded previous day's userDevice state for 2026-01-11. Count: 19997
Loaded current day's raw data for 2026-01-12.
Created current day session summary. Count: 19989
Prepared hardware information. Count: 3500
Prepared applications information. Count: 3500
Prepared update events information. Count: 3500
Joined current day's data. Count: 19989
Aligned current day enriched DataFrame schema.
Union of previous and current day data. Total count: 39986
Filtered for latest states. Final current day userDevice count: 39963
Successfully enriched and saved userDevice state for 2026-01-12 to ../data/applications-gen/enriched/userDevice/2026-01-12
  Finished enrich_user_device_daily for 2026-01-12.
  Calling aggregate_total_n_of_users_daily for 2026-01-12...
Aggregating total dist